In [ ]:
### ----------- initialization ----------- ###
# packages loader + locale
options(encoding = "UTF-8")
Sys.setlocale("LC_ALL", "pl_PL.UTF-8")

use_package <- function(pkg) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
# packages
pkgs <- c("rio","tidyverse","glmnet","pls","caret","leaps","MASS","lars","AER","car","pROC","broom","boot","mgcv", "pacman")
invisible(lapply(pkgs, use_package))

# seed for reproducibility 
set.seed(2025)


# Exercise 1
## Plots annotation
(1) - `MSE of lasso`

(2) - `Lasso path`

## Methodology 
(1) shows how error is changing due to different $\lambda$ parameter (in descending order). First vertical line shows SE, second vertical line shows min 

(2) - Shows lasso path, that is how value of coefficients were changing (in descending order).

## Conclusions
### Ill-conditioning 
From [covariance matrix](#Covariance-matrix) `Condition Index` we see that first 10 Belsley values of columns are below 10. That means that they contain only 1/10 of new information compared to highest eigenvalue. We set this as threshold so we can suspect ill-conditioning in next features. 

### Comparison of PCR and PLS
From summaries and `PCR vs PLS - LOOCV MSEP Comparison`[go to](####PCR-vs-PLS) we see that PLS is much better method, we can obtain best results, already with 6 components where PCR obtains same results at 12 (from 13) components - in this manner we do not reduce any dimensions.


### Model selection 
Best lasso model we obtain with $\lambda$ equals first vertical line from plot (1)[go to](#Plots-exercise-1). It have slightly highest MSE value, but it contains only meaningful variables, which can give ous insides reasoning insides and also reduce overall examination cost of new patients. 

From plot (2)[go to](#Plots-exercise-1) we can see relevance of different variables, and that the most important are features with indices: (6, 3, 13, 1)

In [ ]:
### ----------- Exercise 1: BODYFAT (PCR, PLS, subsets, LASSO) ----------- ###

# parameters
www <- "https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/bodyfat2.xlsx"
local_file <- "bodyfat2.xlsx"

# --------------------------- LOAD DATA ----------------------------------

# Download if file does not exits 
if (!file.exists(local_file)) {
  download.file(www, destfile = local_file, mode = "wb")
}


bodyfat <- rio::import(local_file)

# Print column names (helpful for debugging)
print(colnames(bodyfat))

# Remove missing rows
bodyfat <- dplyr::filter(bodyfat, complete.cases(bodyfat))

# ---------------------- SPLIT X / y -------------------------------------

# Response
y <- bodyfat$bodyfat

# Explanatory variables: remove bodyfat + density
X <- dplyr::select(bodyfat, -bodyfat, -density)

# Matrix for regression methods
Xmat <- stats::model.matrix(~ . - 1, data = X)

#### Covariance matrix

In [ ]:
# ---------------------- Covariance  -------------------------------------

# covariance matrix
cor_matrix <- cor(Xmat)
print('Covariance matrix')
cor_matrix

# eigenvalues of covariance matrix
eigs <- eigen(cor_matrix)$values
print('Eigenvalues')
eigs

# condition index (Belsley)
condition_index <- sqrt(max(eigs) / eigs)
condition_index

plot(condition_index, type = "b", main = "Condition Index", xlab='Eigenvalue idx', ylab='Belsley value of index')


#### PCR vs PLS

In [ ]:
# ------------------ PCR with LOOCV ------------------
print('--------- PCR with LOOCV ---------')
pcr_model <- pcr(
  y ~ Xmat,
  scale = TRUE,
  validation = "LOO"
)

summary(pcr_model) 

pcr_msep <- RMSEP(pcr_model)

# ------------------ PLS with LOOCV ------------------
print('--------- PLS with LOOCV ---------')
pls_model <- plsr(
  y ~ Xmat,
  scale = TRUE,
  validation = "LOO"
)

summary(pls_model)

pls_msep <- RMSEP(pls_model)

print('--------- Comparison plot ---------')
# Wyciągamy MSEP z obiektów pls::RMSEP
pcr_msep_values <- as.vector(pcr_msep$val[1,,])   # MSEP dla PCR
pls_msep_values <- as.vector(pls_msep$val[1,,])   # MSEP dla PLS

# amount of components 
components <- seq_along(pcr_msep_values) - 1

# ------------------ PLT ------------------
# colors
col_Y_pcr <- "blue"
col_X_pcr <- "lightblue"
col_Y_pls <- "red"
col_X_pls <- "pink"


# plot - LOOCV comparison
## plotting 
plot(
  components, pcr_msep_values,
  type = "b", pch = 19, col = col_Y_pcr,
  xlab = "Number of Components",
  ylab = "MSEP (LOOCV)",
  main = "PCR vs PLS – LOOCV MSEP Comparison"
)
lines(components, pls_msep_values, type = "b", pch = 17, col = col_Y_pls)


legend(
  "topright",
  legend = c("PCR", "PLS"),
  col = c(col_Y_pcr, col_Y_pls),
  pch = c(19, 17),
  lty = 1
)

#### Plots exercise 1

In [ ]:
# ---------------------- LASSO REGRESSION (LOOCV) ----------------------

# assigning values
X_lasso <- as.matrix(Xmat)
y_lasso <- y

# model LASSO (alpha = 1)
lasso_cv <- cv.glmnet(
  X_lasso,
  y_lasso,
  alpha = 1,              # LASSO
  nfolds = nrow(X_lasso), # LOOCV
  standardize = TRUE
)

# coefficens for lambda.min and lambda.1se
coef_min  <- coef(lasso_cv, s = "lambda.min")
coef_1se  <- coef(lasso_cv, s = "lambda.1se")

coef_min
coef_1se

# wykres ścieżek LASSO
plot(lasso_cv, main='MSE of lasso')
plot(lasso_cv$glmnet.fit, xvar = "lambda", label = TRUE, main='Lasso path')

# Exercise 2
## Methodology
- we consider different linear models with different penalties
- $\alpha$ in glmnet states which model we use
    - $\alpha = 1$ - lasso (L1 penalty)
    - $\alpha = 0$ - ridge regression (L2 penalty)
    - $\alpha = 0.5$ - elastic net (mix of L1 i L2)

## Results
- From results `Exercise 2 model comparison` does not differ to much. It is because this data are balanced and probable not having many collinear variables. We can select based on what we like to focus on:
    - LASSO: It reduces amount of features - we can select those most important / reduce cost of obtaining data
    - RIGDE: Smallest MAE (optimizing this criterion)
    - OLS: Smallest MSE (optimizing this criterion)
    - ENET: Most balance / stable model - IQR distance in all statistics is small. 


In [ ]:
### ----------- Exercise 2: Boston Housing (OLS, penalised methods) ----------- ###
# parameters
## default local path
local_boston <- "boston_corrected.txt"
url_boston <- "https://www.mimuw.edu.pl/~noble/courses/MultivariateStatistics/data/boston_corrected.txt"

# problem solving
# load file (try local, otherwise download)
if (!file.exists(local_boston)) {
  download.file(url_boston, destfile = local_boston, mode = "wb")
}
boston <- read.table(local_boston, header = FALSE, sep = "\t", fill = TRUE, stringsAsFactors = FALSE)

# naming columns
boston <- boston[, 2:21]
colnames(boston) <- c("TOWN", "TOWN#","TRACT","LON",	"LAT",	"MEDV",	"CMEDV",	"CRIM",	"ZN",	"INDUS",	"CHAS",	"NOX",	"RM",	"AGE",	"DIS",	"RAD",	"TAX",	"PTRATIO",	"B",	"LSTAT")
# taking right columns
regression_cols <- c("CRIM","ZN","INDUS","CHAS","NOX","RM","AGE","DIS","RAD","TAX","PTRATIO","B","LSTAT")
# 
boston <- boston[, c(regression_cols, "MEDV")]
boston <- boston[complete.cases(boston), ]
boston$logMEDV <- log(boston$MEDV)

library(caret)
set.seed(2025)

# caret CV settings (same for all models)
ctrl <- trainControl(
  method = "cv",
  number = 10
)

# ---------------------- OLS ----------------------
train_ols <- train(
  logMEDV ~ .,
  data = boston[, c(regression_cols, "logMEDV")],
  method = "lm",
  trControl = ctrl
)

# ---------------------- LASSO ----------------------
lasso_grid <- expand.grid(
  alpha = 1,                     # LASSO
  lambda = cv_lasso_b$lambda     # reuse glmnet CV grid
)

train_lasso <- train(
  x = Xb,
  y = yb,
  method = "glmnet",
  trControl = ctrl,
  tuneGrid = lasso_grid
)

# ---------------------- Ridge ----------------------
ridge_grid <- expand.grid(
  alpha = 0,                     # Ridge
  lambda = cv_ridge_b$lambda
)

train_ridge <- train(
  x = Xb,
  y = yb,
  method = "glmnet",
  trControl = ctrl,
  tuneGrid = ridge_grid
)

# ---------------------- Elastic Net ----------------------
enet_grid <- expand.grid(
  alpha = 0.5,
  lambda = cv_enet_b$lambda
)

train_enet <- train(
  x = Xb,
  y = yb,
  method = "glmnet",
  trControl = ctrl,
  tuneGrid = enet_grid
)

# ---------------------- Collect results ----------------------
resamps <- resamples(
  list(
    OLS = train_ols,
    LASSO = train_lasso,
    RIDGE = train_ridge,
    ENET = train_enet
  )
)

summary(resamps)
bwplot(resamps, main='Exercise 2 model comparison')

# Exercise 3
## Methodology
- We perform different regression
    - basic poisson regression: $\mathbb{E}[Y|X] = \mu = \exp(\beta_0 + X\beta), \mathrm{Var}[Y|X]=\mu$
    - quasi-poisson: $\mathrm{Var}[Y|X]=\phi\mu$ (we adjust due to dispersion)
    - negative binomial model $\mathrm{Var}(Y) = \mu + \frac{\mu^2}{\theta}$ (we allow dispersion to scale quadratic )
- We check dispersion statistic ($\phi = \frac{\mathrm{Residual Deviance}}{\mathrm{df}}$) of different models, where 
    - dispersion $> 1 \rightarrow$ overdispersion - poisson assumption is violated 
    - dispersion $ < 1 \rightarrow$ underdispersion 

## Results

**Poisson problem**

The Poisson regression does not fit the data satisfactorily.
Although the residual deviance is slightly smaller than the degrees of freedom, the Pearson dispersion statistic φ = 1.33 indicates clear overdispersion. This violates the Poisson assumption Var(Y)=E(Y) and results in underestimated standard errors and artificially low p-values.

**Other methods**

The quasi-Poisson model corrects the standard errors, but it cannot be compared using AIC. The Negative Binomial model introduces a dispersion parameter θ = 0.93 and achieves a substantially lower AIC (6424 vs 6736). This indicates a significantly better fit.

Several variables lose statistical significance in the NB model, suggesting that the Poisson model overstated the evidence due to underestimating variance.

**Conclusion**

We conclude that the Negative Binomial regression is the appropriate model for the DoctorVisits data. The Poisson model is not satisfactory; overdispersion must be accounted for, and the NB model provides the best compromise between fit and interpretability.

In [ ]:
### ----------- Exercise 3: Doctor Visits Data (Poisson Regression) ----------- ###

# ---------------------- Load Data --------------------------------------------

# DoctorVisits is included in AER package
data("DoctorVisits")
dv <- DoctorVisits

# Inspect the dataset
str(dv)
summary(dv)

# Outcome: number of doctor visits (visits)
# Typical explanatory variables: gender, age, income, insurance, etc.

# ---------------------- Fit Poisson Regression -------------------------------

# Basic Poisson regression for count outcome
pois_mod <- glm(
  visits ~ .,        # use all predictors
  data = dv,
  family = poisson(link = "log")
)

summary(pois_mod)

# ---------------------- Check Goodness of Fit --------------------------------

# Residual deviance vs degrees of freedom
pois_mod$deviance
pois_mod$df.residual

# A well-fitting Poisson model should have:
#     Residual deviance ≈ degrees of freedom
#
# If residual deviance >> df → overdispersion present

# ---------------------- Dispersion Statistic ---------------------------------

dispersion <- pois_mod$deviance / pois_mod$df.residual
dispersion

# If dispersion >> 1 → Poisson assumption violated (overdispersion)
# If dispersion < 1 → underdispersion 

# ---------------------- Plot Diagnostics -------------------------------------

par(mfrow = c(2,2))
plot(pois_mod)

# Note: classical residual plots are not ideal for count data;
# interpretation must be cautious.

# ---------------------- Test for Overdispersion ------------------------------

# Using the ratio test
pearson_dispersion <- sum(residuals(pois_mod, type="pearson")^2) / pois_mod$df.residual
pearson_dispersion

# ---------------------- Alternative Models  --------------------------

# 1) Quasi-Poisson (adjusts standard errors)
qpois_mod <- glm(
  visits ~ .,
  data = dv,
  family = quasipoisson(link = "log")
)

summary(qpois_mod)

# 2) Negative Binomial model (handles overdispersion directly)
library(MASS)
nb_mod <- glm.nb(visits ~ ., data = dv)

summary(nb_mod)

# Compare log-likelihood and AIC
AIC(pois_mod, nb_mod)

# ---------------------- Predicted Means Comparison ---------------------------

predict(pois_mod, type = "response")[1:5]
predict(nb_mod, type = "response")[1:5]


# Exercise 4

## Methodology

We consider several statistical models for binary classification of diabetes (“neg” vs “pos”).
The following approaches were used:

### 1. Logistic regression (logit link)
* Models log-odds of diabetes:
  $$(\log \frac{p}{1-p} = \beta_0 + X\beta)$$

### 2. Probit regression
* Models the inverse normal CDF:
  $$( \Phi^{-1}(p) = \beta_0 + X\beta )$$
* Similar to logit but assumes normally distributed latent variable.

### 3. Complementary log-log (cloglog)
* Models asymmetric tail behaviour:
  $$(\log(-\log(1-p)) = \beta_0 + X\beta)$$
* Suitable when the probability of “pos” increases quickly after a threshold.

### 4. Logistic LASSO (glmnet, α = 1)

* Penalised logistic regression with L1 penalty:
  $$( \sum_i \ell_i(\beta) + \lambda \lVert \beta \rVert_1 )$$
* Performs feature selection by shrinking irrelevant coefficients to zero.

### Training methodology
* Complete cases only
* Reference category: `diabetes = "neg"`
* Performance metrics computed:
  * **AIC** (for non-LASSO models)
  * **Classification Accuracy**
  * **AUC** (ROC curve)


## Results

### 1. Model fit (AIC)

**AIC**
* Logistic: **362.02**
* Probit: 362.28  
* Cloglog: 371.93

**Interpretation:**
* **Logistic regression has the lowest AIC**, meaning best goodness-of-fit among standard GLMs.
* Probit is almost identical (AIC difference = 0.25 → negligible).
* **Cloglog performs worse**, meaning its asymmetric link does not match the data well.

---

### 2. Classification Performance (Accuracy, AUC)

**Accuracy**
* logit: 0.783
* probit: 0.783
* cloglog: **0.791** (slightly better)
* LASSO: *same threshold-based performance as logit*

**AUC**
* logit: **0.862**
* probit: 0.861
* cloglog: 0.854
* **LASSO: 0.864 (best)**


**Interpretation:**
* Accuracy is nearly identical across models → no strong difference in classifier behaviour.
* AUC shows slight improvements:
  * **LASSO has the highest AUC**, likely due to regularisation smoothing the coefficients.
  * Logistic / Probit virtually tied.
  * Cloglog slightly worse in ranking ability.

---

## 3. LASSO Path & λ Selection

Two standard λ values:
* **λ.min** → best deviance 
* **λ.1se** → simpler, more stable model

The path plot shows:
* Glucose, mass, pedigree remain strong predictors across λ.
* Insulin and triceps quickly shrink toward zero → low predictive importance.
* This matches the p-values from logit/probit.

---

## Conclusion

### 1. Which GLM link function is best?

* **Logistic regression** gives the best AIC.
* **Probit is equivalent** in practice (difference tiny).
* **Cloglog is worst** → data does not show the skewed behaviour.

### 2. Which classifier predicts best?
* Based on **AUC**, the best model is **LASSO logistic regression**.
* Based on accuracy, all models are similar (≈ 0.78–0.79).

### 3. Which features matter most? (Consistency across models)
Strong predictors:
* **glucose** (highly significant in all models)
* **mass (BMI)**
* **pedigree**

Weak predictors consistently:
* pressure
* triceps
* insulin
(Exactly what LASSO shrinks.)
### 4. Recommended model
* **LASSO logistic regression**: best AUC + feature selection
* **Logistic regression**: best AIC + interpretable
* Probit and cloglog do not improve performance

In [ ]:
### ----------- Exercise 4: Diabetes (Logistic, Probit, Extreme Value, LASSO) ----------- ###

# ---------------------- Load Data ------------------------------------------
pacman::p_load(tidyverse, mlbench, broom, pander)
data("PimaIndiansDiabetes2", package = "mlbench")
# Remove incomplete observations and set reference level for response
pid2 <- PimaIndiansDiabetes2 %>%
  filter(complete.cases(.)) %>% 
  mutate(diabetes = relevel(diabetes, ref = "neg"))

str(pid2)
summary(pid2)

# ---------------------- Logistic Regression --------------------------------

logit_mod <- glm(
  diabetes ~ ., 
  data = pid2,
  family = binomial(link = "logit")
)

summary(logit_mod)

# ---------------------- Probit Regression ----------------------------------

probit_mod <- glm(
  diabetes ~ ., 
  data = pid2,
  family = binomial(link = "probit")
)

summary(probit_mod)

# ---------------------- Extreme-Value (Complementary log-log) ---------------

cloglog_mod <- glm(
  diabetes ~ ., 
  data = pid2,
  family = binomial(link = "cloglog")
)

summary(cloglog_mod)

# ---------------------- Model Comparison: AIC -------------------------------

AIC(logit_mod, probit_mod, cloglog_mod)

# ---------------------- Prediction Performance ------------------------------

# Function to compute accuracy and AUC
model_eval <- function(model, data) {
  preds <- predict(model, type = "response")
  pred_class <- ifelse(preds > 0.5, "pos", "neg")

  accuracy <- mean(pred_class == data$diabetes)
  auc <- roc(data$diabetes, preds)$auc

  return(list(accuracy = accuracy, auc = auc))
}

eval_logit  <- model_eval(logit_mod, pid2)
eval_probit <- model_eval(probit_mod, pid2)
eval_cloglog <- model_eval(cloglog_mod, pid2)

eval_logit
eval_probit
eval_cloglog

# ---------------------- LASSO Logistic Regression ---------------------------

# Build design matrix
X <- model.matrix(diabetes ~ . - 1, data = pid2)  # no intercept
y <- ifelse(pid2$diabetes == "pos", 1, 0)         # convert to 0/1

# LASSO logistic (alpha = 1)
lasso_cv <- cv.glmnet(
  X, y,
  alpha = 1,
  family = "binomial"
)

plot(lasso_cv)   # CV curve
plot(lasso_cv$glmnet.fit, xvar = "lambda", label = TRUE)  # regularization path

# Best lambda
lasso_cv$lambda.min
lasso_cv$lambda.1se

# Coefficients at optimal lambda
coef_lasso <- coef(lasso_cv, s = "lambda.min")
coef_lasso

# Predict
lasso_preds <- predict(lasso_cv, newx = X, s = "lambda.min", type = "response")
lasso_class <- ifelse(lasso_preds > 0.5, "pos", "neg")

lasso_accuracy <- mean(lasso_class == pid2$diabetes)
lasso_auc <- roc(pid2$diabetes, as.numeric(lasso_preds))$auc

lasso_accuracy
lasso_auc

# ---------------------- Final Comparison Table ------------------------------

results <- tibble(
  Model = c("Logistic", "Probit", "Cloglog", "LASSO Logistic"),
  AIC = c(
    AIC(logit_mod),
    AIC(probit_mod),
    AIC(cloglog_mod),
    NA  # cv.glmnet has no AIC
  ),
  Accuracy = c(
    eval_logit$accuracy,
    eval_probit$accuracy,
    eval_cloglog$accuracy,
    lasso_accuracy
  ),
  AUC = c(
    eval_logit$auc,
    eval_probit$auc,
    eval_cloglog$auc,
    lasso_auc
  )
)

pander::pander(results)
